# Event-Driven Signals Analysis (20 Trading Days)

This notebook provides interactive analysis of **event-driven signals only** from the OptiCore trading bot.

**Data Summary:**
- **Total Event Signals**: 1,051 signals (20 trading days: 2026-01-20 → 2026-02-17)
- **Signals by Type**: Buy (505), Neutral (546), Sell (0)
- **Top Event Type**: Volatility Expansion (48.8%)

**Steps:**
1. Load the CSV reports generated on EC2
2. Analyze signal distribution and daily trends
3. Examine per-ticker bias and event-type breakdown
4. Interactive filtering by date, ticker, and event type

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter

# Load event-driven signals report
OUTDIR = Path('../reports/event_signals_bias_20trading')

print('Loading event-driven signals report...')
print(f'Looking for CSVs in: {OUTDIR.resolve()}')

# Load all CSVs
signals_df = pd.read_csv(OUTDIR / 'event_signals_raw_20trading.csv')
signal_counts = pd.read_csv(OUTDIR / 'event_signal_counts_20trading.csv')
daily_counts = pd.read_csv(OUTDIR / 'event_daily_signal_counts.csv')
per_symbol = pd.read_csv(OUTDIR / 'event_per_symbol_bias.csv')
event_types = pd.read_csv(OUTDIR / 'event_type_distribution.csv')

print(f'✅ Loaded {len(signals_df)} event-driven signals')
print(f'\n📊 Report Files:')
for f in OUTDIR.glob('*.csv'):
    print(f'  - {f.name}')

# Display summary
print(f'\n📈 Signal Distribution:')
print(signal_counts.to_string(index=False))

print(f'\n⚡ Top 10 Event Types:')
print(event_types.sort_values('count', ascending=False).head(10).to_string(index=False))

print(f'\n🎯 Top 10 Tickers by %BUY:')
print(per_symbol.sort_values('pct_buy', ascending=False).head(10).to_string(index=False))

In [ ]:
# Daily Signal Trends (last 20 days)

print('📅 Daily Event-Driven Signal Counts (recent 15 days):')
print(daily_counts.sort_values('day', ascending=False).head(15).to_string(index=False))

# Calculate rolling stats
daily_counts_sorted = daily_counts.sort_values('day')
daily_counts_sorted['avg_buy'] = daily_counts_sorted['buy'].rolling(5, min_periods=1).mean()
daily_counts_sorted['avg_total'] = daily_counts_sorted['total'].rolling(5, min_periods=1).mean()

print(f'\n📊 5-day rolling average:')
print(daily_counts_sorted[['day', 'buy', 'total', 'avg_buy', 'avg_total']].sort_values('day', ascending=False).head(10).to_string(index=False))

In [ ]:
# Event Type Breakdown with Per-Event Buy%

# Extract all unique event types and their signal composition
signals_df['signal_label'] = signals_df['signal'].map({1: 'buy', -1: 'sell', 0: 'neutral'})
signals_df['event_type'] = signals_df['triggered_by']

event_stats = signals_df.groupby('event_type').agg({
    'signal_label': lambda x: (x == 'buy').sum(),
    'id': 'count'
}).rename(columns={'signal_label': 'buy_count', 'id': 'total'})

event_stats['pct_buy'] = (event_stats['buy_count'] / event_stats['total'] * 100).round(1)
event_stats = event_stats.sort_values('total', ascending=False)

print('⚡ Event Type Buy% Breakdown:')
print(event_stats.to_string())

print(f'\n📊 Key Insights:')
print(f'  - Most common: {event_stats.index[0]} ({event_stats.iloc[0]["total"]} signals)')
print(f'  - Highest %BUY: {event_stats["pct_buy"].idxmax()} ({event_stats["pct_buy"].max():.1f}%)')
print(f'  - Lowest %BUY: {event_stats["pct_buy"].idxmin()} ({event_stats["pct_buy"].min():.1f}%)')

In [ ]:
# Ticker Analysis: Event Types per Ticker

# Show which event types are most active per ticker
ticker_event_dist = signals_df.groupby(['ticker', 'event_type']).size().unstack(fill_value=0)

print('📍 Event Type Distribution by Ticker (Top 10 Tickers by Signal Count):')
top_tickers = signals_df['ticker'].value_counts().head(10).index
ticker_event_subset = ticker_event_dist.loc[top_tickers]
print(ticker_event_subset.to_string())

# Highlight ticker biases
print(f'\n🎯 Ticker Buy% Rankings (event-driven):')
ticker_buy_pct = signals_df[signals_df['signal_label'] == 'buy'].groupby('ticker').size() / signals_df.groupby('ticker').size() * 100
ticker_buy_pct = ticker_buy_pct.sort_values(ascending=False)
print(ticker_buy_pct.round(1).to_string())

In [ ]:
# Interactive Filter: Examine Specific Signals

# Example: Show recent BUY signals from volatility_expansion
print('🔍 Recent BUY signals from Volatility Expansion events:\n')

recent_buys = signals_df[
    (signals_df['signal'] == 1) & 
    (signals_df['event_type'] == 'event:volatility_expansion')
].sort_values('timestamp', ascending=False).head(10)

if len(recent_buys) > 0:
    print(recent_buys[['timestamp', 'ticker', 'interval', 'signal_label', 'confidence', 'event_type']].to_string(index=False))
else:
    print('No BUY signals from volatility_expansion found.')

# Example: Show EURGBP signals (highest %BUY ticker)
print('\n\n📊 All EURGBP event-driven signals (highest %BUY = 100%):\n')
eurgbp = signals_df[signals_df['ticker'] == 'EURGBP'].sort_values('timestamp', ascending=False)
print(f'Total EURGBP event signals: {len(eurgbp)}')
print(eurgbp[['timestamp', 'interval', 'signal_label', 'confidence', 'event_type']].head(15).to_string(index=False))

# Distribution
print(f'\n  Signal breakdown:')
print(f'    - BUY: {(eurgbp["signal"] == 1).sum()}')
print(f'    - NEUTRAL: {(eurgbp["signal"] == 0).sum()}')
print(f'    - SELL: {(eurgbp["signal"] == -1).sum()}')

In [ ]:
# Confidence Analysis & Summary Statistics

print('📈 Confidence Statistics (Event-Driven Signals):\n')

# Overall confidence
print(f'Overall Confidence:')
print(f'  Mean: {signals_df["confidence"].mean():.4f}')
print(f'  Median: {signals_df["confidence"].median():.4f}')
print(f'  Std Dev: {signals_df["confidence"].std():.4f}')
print(f'  Min: {signals_df["confidence"].min():.4f}')
print(f'  Max: {signals_df["confidence"].max():.4f}')

# By signal type
print(f'\nConfidence by Signal Type:')
for signal_label in ['buy', 'neutral']:
    subset = signals_df[signals_df['signal_label'] == signal_label]
    if len(subset) > 0:
        print(f'  {signal_label.upper()}: mean={subset["confidence"].mean():.4f}, median={subset["confidence"].median():.4f} (n={len(subset)})')

# By top event types
print(f'\nConfidence by Top 5 Event Types:')
top_events = signals_df['event_type'].value_counts().head(5).index
for event in top_events:
    subset = signals_df[signals_df['event_type'] == event]
    print(f'  {event}: mean={subset["confidence"].mean():.4f}, n={len(subset)}')

print(f'\n✅ Summary:')
print(f'  Total event-driven signals: {len(signals_df)}')
print(f'  Date range: {signals_df["timestamp"].min()} to {signals_df["timestamp"].max()}')
print(f'  Unique symbols: {signals_df["ticker"].nunique()}')
print(f'  Unique event types: {signals_df["event_type"].nunique()}')